# Scenario: Catching Malfunctioning IoT Oxygen Sensors

In [3]:
import pandas as pd
import sqlite3
# creating dataset representing streaming vitals from home sensors
sensor_data = {
    "reading_id": [3001, 3002, 3003, 3004, 3005, 3006, 3007],
    "patient_id": ["P-50", "P-51", "P-52", "P-53", "P-54", "P-55", "P-56"],
    "spo2_percentage": [98, 96, 0, 94, 999, 88, 97], # Note the 0 and 999!
    "device_model": ["Oxifit-v1", "Oxifit-v1", "PulseMax-3", "Oxifit-v1", "PulseMax-3", "Oxifit-v1", "PulseMax-3"]    
}
# adding dataset to DataFrame
df_sensor_data = pd.DataFrame(sensor_data)
# conneting the sql to save the dataframe in temp memory
connt = sqlite3.connect(":memory:")
df_sensor_data.to_sql("sensor_data_record", connt,  index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("******************************* Sensor Boundary Audit Database is ready! ******************")

******************************* Sensor Boundary Audit Database is ready! ******************


# The Impossible Outlier Scan

In [10]:
# query for all data to review 
all_data = "SELECT * FROM sensor_data_record"
print("*************************************** all_ data to review *******************")
display(run_query(all_data))
# query to isolate any records where the spo2_percentage is biologically impossible (less than or equal to 0 OR greater than 100).
mulfunction_records = """
SELECT patient_id, spo2_percentage, device_model 
FROM sensor_data_record
WHERE spo2_percentage <= 0 OR spo2_percentage > 100
"""
print("********************************** records where the spo2_percentage is biologically impossible *************")
display(run_query(mulfunction_records))

*************************************** all_ data to review *******************


,reading_id,patient_id,spo2_percentage,device_model
0,3001,P-50,98,Oxifit-v1
1,3002,P-51,96,Oxifit-v1
2,3003,P-52,0,PulseMax-3
3,3004,P-53,94,Oxifit-v1
4,3005,P-54,999,PulseMax-3
5,3006,P-55,88,Oxifit-v1
6,3007,P-56,97,PulseMax-3


********************************** records where the spo2_percentage is biologically impossible *************


,patient_id,spo2_percentage,device_model
0,P-52,0,PulseMax-3
1,P-54,999,PulseMax-3


# Hardware Malfunction Profiler

In [11]:
# query to count how many impossible readings (out of the 0–100 range) are occurring, grouped by device_model
total_impossible_reading = """
SELECT device_model,
       COUNT(*) as currupt_reading_count
FROM sensor_data_record
WHERE  spo2_percentage <= 0 OR spo2_percentage > 100
GROUP BY device_model;
"""
print("********************************** malfunctioned hardware has been found! ************")
display(run_query(total_impossible_reading))

********************************** malfunctioned hardware has been found! ************


,device_model,currupt_reading_count
0,PulseMax-3,2
